# 02 - Feature Engineering

## Objectif

Construire les variables explicatives nécessaires au modèle de
propensity scoring client × produit.

Le modèle devra répondre à la question :

> "Quelle est la probabilité qu'un client achète à nouveau un produit ?"

### Principe important

Les features doivent être calculées uniquement à partir de l'historique
disponible AVANT la commande cible.

Nous utilisons donc ici :

- `orders.csv`
- `order_products__prior.csv`
- `products.csv`
- `aisles.csv`
- `departments.csv`

Nous n'utilisons PAS :

- `order_products__train.csv`

Le fichier `order_products__train.csv` sera utilisé plus tard pour construire
la variable cible dans `03_build_training_set.ipynb`.

### Niveaux de features

1. Client
2. Produit
3. Client × Produit
4. Client × Département

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("Pandas version:", pd.__version__)

Pandas version: 2.2.1


In [5]:
# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path.cwd().parent

# Si le notebook est exécuté depuis la racine du projet,
# utiliser plutôt :
# PROJECT_DIR = Path.cwd()

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory :", PROJECT_DIR)
print("Raw directory     :", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)

Project directory : c:\Users\djuff\OneDrive\Drive_perso_ceed\OneDrive\Documents\CDI
Raw directory     : c:\Users\djuff\OneDrive\Drive_perso_ceed\OneDrive\Documents\CDI\data\raw
Processed directory: c:\Users\djuff\OneDrive\Drive_perso_ceed\OneDrive\Documents\CDI\data\processed


In [8]:
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

# Chemin absolu de ton projet
PROJECT_DIR = Path(
    r"C:\Users\djuff\OneDrive\Drive_perso_ceed\OneDrive\Documents\CDI\instacart_propensity"
)

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

# Créer processed s'il n'existe pas
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR :", PROJECT_DIR)
print("RAW_DIR     :", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_DIR : C:\Users\djuff\OneDrive\Drive_perso_ceed\OneDrive\Documents\CDI\instacart_propensity
RAW_DIR     : C:\Users\djuff\OneDrive\Drive_perso_ceed\OneDrive\Documents\CDI\instacart_propensity\data\raw
PROCESSED_DIR: C:\Users\djuff\OneDrive\Drive_perso_ceed\OneDrive\Documents\CDI\instacart_propensity\data\processed


In [10]:
# ============================================================
# CHECK RAW FILES
# ============================================================

print("Fichiers présents dans data/raw :")

for file in RAW_DIR.iterdir():
    print(" -", file.name)

Fichiers présents dans data/raw :
 - aisles.csv
 - departments.csv
 - orders.csv
 - order_products__prior.csv
 - order_products__train.csv
 - products.csv


In [12]:
orders = pd.read_csv(RAW_DIR / "orders.csv")

products = pd.read_csv(RAW_DIR / "products.csv")

aisles = pd.read_csv(RAW_DIR / "aisles.csv")

departments = pd.read_csv(RAW_DIR / "departments.csv")

order_products_prior = pd.read_csv(
    RAW_DIR / "order_products__prior.csv"
)

order_products_train = pd.read_csv(
    RAW_DIR / "order_products__train.csv"
)

print("orders :", orders.shape)
print("products :", products.shape)
print("aisles :", aisles.shape)
print("departments :", departments.shape)
print("order_products_prior :", order_products_prior.shape)
print("order_products_train :", order_products_train.shape)

orders : (3421083, 7)
products : (49688, 4)
aisles : (134, 2)
departments : (21, 2)
order_products_prior : (32434489, 4)
order_products_train : (1384617, 4)


In [13]:
# Aperçu des données

display(orders.head())
display(order_products_prior.head())
display(products.head())

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [14]:
# ============================================================
# DATA QUALITY CHECK
# ============================================================

print("Orders")
print(orders.info())

print("\nProducts")
print(products.info())

print("\nOrder products prior")
print(order_products_prior.info())

Orders
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 7 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   user_id                 int64  
 2   eval_set                object 
 3   order_number            int64  
 4   order_dow               int64  
 5   order_hour_of_day       int64  
 6   days_since_prior_order  float64
dtypes: float64(1), int64(5), object(1)
memory usage: 182.7+ MB
None

Products
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49688 entries, 0 to 49687
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   product_id     49688 non-null  int64 
 1   product_name   49688 non-null  object
 2   aisle_id       49688 non-null  int64 
 3   department_id  49688 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 1.5+ MB
None

Order products prior
<class 'pandas.core.frame.

In [15]:
# Valeurs manquantes

print("Missing values - orders")
display(orders.isna().sum())

print("\nMissing values - order_products_prior")
display(order_products_prior.isna().sum())

print("\nMissing values - products")
display(products.isna().sum())

Missing values - orders


order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64


Missing values - order_products_prior


order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64


Missing values - products


product_id       0
product_name     0
aisle_id         0
department_id    0
dtype: int64

# Construction de l'historique

Nous conservons uniquement les commandes `prior`.

Pourquoi ?

Parce que la commande `train` représente la commande future que nous
souhaitons prédire.

Utiliser les produits de cette commande pour construire les features
introduirait une fuite de données (data leakage).

In [16]:
# ============================================================
# PRIOR ORDERS ONLY
# ============================================================

orders_prior = orders[
    orders["eval_set"] == "prior"
].copy()

print("Nombre de commandes prior :", len(orders_prior))
print("Nombre de clients :", orders_prior["user_id"].nunique())

Nombre de commandes prior : 3214874
Nombre de clients : 206209


In [17]:
# ============================================================
# MERGE ORDERS + PRODUCTS
# ============================================================

history = order_products_prior.merge(
    orders_prior[
        [
            "order_id",
            "user_id",
            "order_number",
            "order_dow",
            "order_hour_of_day",
            "days_since_prior_order"
        ]
    ],
    on="order_id",
    how="inner"
)

print("History shape:", history.shape)

display(history.head())

History shape: (32434489, 9)


,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2,33120,1,1,202279,3,5,9,8.0
1,2,28985,2,1,202279,3,5,9,8.0
2,2,9327,3,0,202279,3,5,9,8.0
3,2,45918,4,1,202279,3,5,9,8.0
4,2,30035,5,0,202279,3,5,9,8.0


# 5. Customer Features

Ces variables décrivent le comportement général du client :

- nombre de commandes
- diversité des produits achetés
- diversité des rayons
- taille moyenne du panier
- fréquence d'achat
- jour préféré
- heure préférée

In [18]:
# ============================================================
# CUSTOMER FEATURES
# ============================================================

customer_features = (
    orders_prior
    .groupby("user_id")
    .agg(
        total_orders=("order_number", "max"),
        avg_days_between_orders=("days_since_prior_order", "mean"),
        median_days_between_orders=("days_since_prior_order", "median"),
        preferred_day_of_week=("order_dow", lambda x: x.mode().iloc[0]),
        preferred_hour=("order_hour_of_day", lambda x: x.mode().iloc[0])
    )
    .reset_index()
)

customer_features.head()

,user_id,total_orders,avg_days_between_orders,median_days_between_orders,preferred_day_of_week,preferred_hour
0,1,10,19.555556,20.0,1,7
1,2,14,15.230769,13.0,1,10
2,3,12,12.090909,11.0,0,16
3,4,5,13.750000,17.0,4,11
4,5,4,13.333333,11.0,3,18


In [19]:
# ============================================================
# CUSTOMER PRODUCT DIVERSITY
# ============================================================

customer_diversity = (
    history
    .groupby("user_id")
    .agg(
        unique_products=("product_id", "nunique"),
        unique_aisles=("product_id", "nunique"),
        unique_departments=("product_id", "nunique")
    )
    .reset_index()
)

In [20]:
# ============================================================
# ADD PRODUCT INFORMATION
# ============================================================

history = history.merge(
    products[
        [
            "product_id",
            "aisle_id",
            "department_id"
        ]
    ],
    on="product_id",
    how="left"
)

print(history.shape)

display(history.head())

(32434489, 11)


,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,aisle_id,department_id
0,2,33120,1,1,202279,3,5,9,8.0,86,16
1,2,28985,2,1,202279,3,5,9,8.0,83,4
2,2,9327,3,0,202279,3,5,9,8.0,104,13
3,2,45918,4,1,202279,3,5,9,8.0,19,13
4,2,30035,5,0,202279,3,5,9,8.0,17,13


In [21]:
# ============================================================
# CORRECT CUSTOMER DIVERSITY FEATURES
# ============================================================

customer_diversity = (
    history
    .groupby("user_id")
    .agg(
        unique_products=("product_id", "nunique"),
        unique_aisles=("aisle_id", "nunique"),
        unique_departments=("department_id", "nunique")
    )
    .reset_index()
)

customer_diversity.head()

,user_id,unique_products,unique_aisles,unique_departments
0,1,18,12,7
1,2,102,33,13
2,3,33,16,9
3,4,17,14,9
4,5,23,16,9


In [22]:
# ============================================================
# AVERAGE BASKET SIZE
# ============================================================

basket_size = (
    history
    .groupby(["user_id", "order_id"])
    .size()
    .groupby("user_id")
    .mean()
    .reset_index(name="avg_products_per_order")
)

basket_size.head()

,user_id,avg_products_per_order
0,1,5.900000
1,2,13.928571
2,3,7.333333
3,4,3.600000
4,5,9.250000


In [23]:
# ============================================================
# MERGE CUSTOMER FEATURES
# ============================================================

customer_features = (
    customer_features
    .merge(
        customer_diversity,
        on="user_id",
        how="left"
    )
    .merge(
        basket_size,
        on="user_id",
        how="left"
    )
)

customer_features.head()

,user_id,total_orders,avg_days_between_orders,median_days_between_orders,preferred_day_of_week,preferred_hour,unique_products,unique_aisles,unique_departments,avg_products_per_order
0,1,10,19.555556,20.0,1,7,18,12,7,5.900000
1,2,14,15.230769,13.0,1,10,102,33,13,13.928571
2,3,12,12.090909,11.0,0,16,33,16,9,7.333333
3,4,5,13.750000,17.0,4,11,17,14,9,3.600000
4,5,4,13.333333,11.0,3,18,23,16,9,9.250000


In [24]:
# ============================================================
# MERGE CUSTOMER FEATURES
# ============================================================

customer_features = (
    customer_features
    .merge(
        customer_diversity,
        on="user_id",
        how="left"
    )
    .merge(
        basket_size,
        on="user_id",
        how="left"
    )
)

customer_features.head()

,user_id,total_orders,avg_days_between_orders,median_days_between_orders,preferred_day_of_week,preferred_hour,unique_products_x,unique_aisles_x,unique_departments_x,avg_products_per_order_x,unique_products_y,unique_aisles_y,unique_departments_y,avg_products_per_order_y
0,1,10,19.555556,20.0,1,7,18,12,7,5.900000,18,12,7,5.900000
1,2,14,15.230769,13.0,1,10,102,33,13,13.928571,102,33,13,13.928571
2,3,12,12.090909,11.0,0,16,33,16,9,7.333333,33,16,9,7.333333
3,4,5,13.750000,17.0,4,11,17,14,9,3.600000,17,14,9,3.600000
4,5,4,13.333333,11.0,3,18,23,16,9,9.250000,23,16,9,9.250000


In [25]:
# ============================================================
# CUSTOMER REORDER BEHAVIOR
# ============================================================

customer_reorder = (
    history
    .groupby("user_id")
    .agg(
        customer_reorder_ratio=("reordered", "mean")
    )
    .reset_index()
)

customer_features = customer_features.merge(
    customer_reorder,
    on="user_id",
    how="left"
)

display(customer_features.head())

,user_id,total_orders,avg_days_between_orders,median_days_between_orders,preferred_day_of_week,preferred_hour,unique_products_x,unique_aisles_x,unique_departments_x,avg_products_per_order_x,unique_products_y,unique_aisles_y,unique_departments_y,avg_products_per_order_y,customer_reorder_ratio
0,1,10,19.555556,20.0,1,7,18,12,7,5.900000,18,12,7,5.900000,0.694915
1,2,14,15.230769,13.0,1,10,102,33,13,13.928571,102,33,13,13.928571,0.476923
2,3,12,12.090909,11.0,0,16,33,16,9,7.333333,33,16,9,7.333333,0.625000
3,4,5,13.750000,17.0,4,11,17,14,9,3.600000,17,14,9,3.600000,0.055556
4,5,4,13.333333,11.0,3,18,23,16,9,9.250000,23,16,9,9.250000,0.378378


# 6. Product Features

Ces variables mesurent la popularité et le comportement de réachat
associé à chaque produit.

In [26]:
# ============================================================
# PRODUCT FEATURES
# ============================================================

product_features = (
    history
    .groupby("product_id")
    .agg(
        product_purchase_count=("order_id", "count"),
        product_unique_users=("user_id", "nunique"),
        product_reorder_rate=("reordered", "mean")
    )
    .reset_index()
)

product_features.head()

,product_id,product_purchase_count,product_unique_users,product_reorder_rate
0,1,1852,716,0.613391
1,2,90,78,0.133333
2,3,277,74,0.732852
3,4,329,182,0.446809
4,5,15,6,0.600000


In [27]:
# ============================================================
# PRODUCT POPULARITY RANK
# ============================================================

product_features["product_popularity_rank"] = (
    product_features["product_purchase_count"]
    .rank(
        ascending=False,
        method="min"
    )
)

product_features.head()

,product_id,product_purchase_count,product_unique_users,product_reorder_rate,product_popularity_rank
0,1,1852,716,0.613391,2996.0
1,2,90,78,0.133333,20886.0
2,3,277,74,0.732852,11948.0
3,4,329,182,0.446809,10728.0
4,5,15,6,0.600000,38159.0


# 7. Customer × Product Features

C'est la partie la plus importante du projet de propensity scoring.

Nous cherchons à caractériser la relation entre :

    CLIENT × PRODUIT

Exemples :

- Combien de fois le client a acheté le produit ?
- Quand l'a-t-il acheté pour la dernière fois ?
- Quelle proportion de ses commandes contient ce produit ?
- Achète-t-il régulièrement ce produit ?
- Depuis combien de commandes ne l'a-t-il pas acheté ?
- A-t-il une série de réachats consécutifs ?

In [28]:
# ============================================================
# CUSTOMER × PRODUCT
# ============================================================

customer_product_features = (
    history
    .groupby(["user_id", "product_id"])
    .agg(
        customer_product_purchase_count=("order_id", "count"),
        customer_product_reorder_rate=("reordered", "mean"),
        first_order_number=("order_number", "min"),
        last_order_number=("order_number", "max"),
        share_of_customer_orders_containing_product=("order_id", "nunique")
    )
    .reset_index()
)

customer_product_features.head()

,user_id,product_id,customer_product_purchase_count,customer_product_reorder_rate,first_order_number,last_order_number,share_of_customer_orders_containing_product
0,1,196,10,0.900000,1,10,10
1,1,10258,9,0.888889,2,10,9
2,1,10326,1,0.000000,5,5,1
3,1,12427,10,0.900000,1,10,10
4,1,13032,3,0.666667,2,10,3


In [29]:
# ============================================================
# ORDERS SINCE LAST PURCHASE
# ============================================================

customer_last_order = (
    orders_prior
    .groupby("user_id")["order_number"]
    .max()
    .reset_index(name="customer_last_order_number")
)

customer_product_features = customer_product_features.merge(
    customer_last_order,
    on="user_id",
    how="left"
)

customer_product_features["orders_since_last_purchase"] = (
    customer_product_features["customer_last_order_number"]
    - customer_product_features["last_order_number"]
)

customer_product_features.head()

,user_id,product_id,customer_product_purchase_count,customer_product_reorder_rate,first_order_number,last_order_number,share_of_customer_orders_containing_product,customer_last_order_number,orders_since_last_purchase
0,1,196,10,0.900000,1,10,10,10,0
1,1,10258,9,0.888889,2,10,9,10,0
2,1,10326,1,0.000000,5,5,1,10,5
3,1,12427,10,0.900000,1,10,10,10,0
4,1,13032,3,0.666667,2,10,3,10,0


In [30]:
# ============================================================
# SHARE OF CUSTOMER ORDERS
# ============================================================

customer_product_features["share_of_customer_orders_containing_product"] = (
    customer_product_features["customer_product_purchase_count"]
    / customer_product_features["customer_last_order_number"]
)

customer_product_features.head()

,user_id,product_id,customer_product_purchase_count,customer_product_reorder_rate,first_order_number,last_order_number,share_of_customer_orders_containing_product,customer_last_order_number,orders_since_last_purchase
0,1,196,10,0.900000,1,10,1.0,10,0
1,1,10258,9,0.888889,2,10,0.9,10,0
2,1,10326,1,0.000000,5,5,0.1,10,5
3,1,12427,10,0.900000,1,10,1.0,10,0
4,1,13032,3,0.666667,2,10,0.3,10,0


In [31]:
# ============================================================
# AVERAGE ORDERS BETWEEN PURCHASES
# ============================================================

purchase_orders = (
    history[
        [
            "user_id",
            "product_id",
            "order_number"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["user_id", "product_id", "order_number"]
    )
)

purchase_orders["order_gap"] = (
    purchase_orders
    .groupby(["user_id", "product_id"])["order_number"]
    .diff()
)

avg_order_gap = (
    purchase_orders
    .groupby(["user_id", "product_id"])["order_gap"]
    .mean()
    .reset_index(name="avg_orders_between_purchases")
)

customer_product_features = customer_product_features.merge(
    avg_order_gap,
    on=["user_id", "product_id"],
    how="left"
)

customer_product_features.head()

,user_id,product_id,customer_product_purchase_count,customer_product_reorder_rate,first_order_number,last_order_number,share_of_customer_orders_containing_product,customer_last_order_number,orders_since_last_purchase,avg_orders_between_purchases
0,1,196,10,0.900000,1,10,1.0,10,0,1.0
1,1,10258,9,0.888889,2,10,0.9,10,0,1.0
2,1,10326,1,0.000000,5,5,0.1,10,5,NaN
3,1,12427,10,0.900000,1,10,1.0,10,0,1.0
4,1,13032,3,0.666667,2,10,0.3,10,0,4.0


# 8. Customer × Department Features

Ces variables permettent de mesurer l'affinité d'un client pour
les différents départements.

Exemple :

Un client achète :

- 40 % de produits "Dairy Eggs"
- 25 % de produits "Produce"
- 15 % de produits "Snacks"

On peut alors utiliser son affinité pour un département comme
signal de cross-sell.

In [34]:
# ============================================================
# CUSTOMER × DEPARTMENT
# ============================================================

customer_department_features = (
    history
    .groupby(["user_id", "department_id"])
    .agg(
        customer_department_purchase_count=(
            "product_id",
            "count"
        )
    )
    .reset_index()
)

customer_department_features.head()

,user_id,department_id,customer_department_purchase_count
0,1,4,5
1,1,7,13
2,1,13,1
3,1,14,3
4,1,16,13


In [ ]:
# ============================================================
# CUSTOMER TOTAL PRODUCT PURCHASES
# ============================================================

customer_total_purchases = (
    history
    .groupby("user_id")
    .size()
    .reset_index(
        name="customer_total_product_purchases"
    )
)

customer_department_features = (
    customer_department_features
    .merge(
        customer_total_purchases,
        on="user_id",
        how="left"
    )
)

customer_department_features[
    "customer_department_share"
] = (
    customer_department_features[
        "customer_department_purchase_count"
    ]
    /
    customer_department_features[
        "customer_total_product_purchases"
    ]
)

customer_department_features.head()

,user_id,department_id,customer_department_purchase_count,customer_total_product_purchases,customer_department_share
0,1,4,5,59,0.084746
1,1,7,13,59,0.220339
2,1,13,1,59,0.016949
3,1,14,3,59,0.050847
4,1,16,13,59,0.220339


In [36]:
# ============================================================
# DEPARTMENT LABEL
# ============================================================

customer_department_features = (
    customer_department_features
    .merge(
        departments,
        on="department_id",
        how="left"
    )
)

customer_department_features.head()

,user_id,department_id,customer_department_purchase_count,customer_total_product_purchases,customer_department_share,department
0,1,4,5,59,0.084746,produce
1,1,7,13,59,0.220339,beverages
2,1,13,1,59,0.016949,pantry
3,1,14,3,59,0.050847,breakfast
4,1,16,13,59,0.220339,dairy eggs


In [37]:
# ============================================================
# PRODUCT METADATA
# ============================================================

product_features = (
    product_features
    .merge(
        products[
            [
                "product_id",
                "product_name",
                "aisle_id",
                "department_id"
            ]
        ],
        on="product_id",
        how="left"
    )
)

product_features = (
    product_features
    .merge(
        aisles,
        on="aisle_id",
        how="left"
    )
)

product_features = (
    product_features
    .merge(
        departments,
        on="department_id",
        how="left"
    )
)

product_features.head()

,product_id,product_purchase_count,product_unique_users,product_reorder_rate,product_popularity_rank,product_name,aisle_id,department_id,aisle,department
0,1,1852,716,0.613391,2996.0,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,90,78,0.133333,20886.0,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,277,74,0.732852,11948.0,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages
3,4,329,182,0.446809,10728.0,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen meals,frozen
4,5,15,6,0.600000,38159.0,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry


In [38]:
# ============================================================
# DATA QUALITY - CUSTOMER PRODUCT
# ============================================================

print(
    "Nombre de lignes :",
    len(customer_product_features)
)

print(
    "Nombre de clients :",
    customer_product_features["user_id"].nunique()
)

print(
    "Nombre de produits :",
    customer_product_features["product_id"].nunique()
)

print(
    "Nombre de couples client-produit :",
    customer_product_features[
        ["user_id", "product_id"]
    ].drop_duplicates().shape[0]
)

Nombre de lignes : 13307953
Nombre de clients : 206209
Nombre de produits : 49677
Nombre de couples client-produit : 13307953


In [39]:
# Vérification des doublons

duplicates = (
    customer_product_features
    .duplicated(
        subset=["user_id", "product_id"]
    )
    .sum()
)

print("Doublons client × produit :", duplicates)

Doublons client × produit : 0


In [42]:
# ============================================================
# CHECK NUMERIC FEATURES
# ============================================================

numeric_columns = [
    "customer_product_purchase_count",
    "customer_product_reorder_rate",
    "orders_since_last_purchase",
    "share_of_customer_orders_containing_product",
    "avg_orders_between_purchases",
    "product_streak"
]



In [44]:
# ============================================================
# RATE CHECK
# ============================================================

print(
    "Customer-product reorder rate:"
)

print(
    customer_product_features[
        "customer_product_reorder_rate"
    ].describe()
)

print(
    "\nCustomer order share:"
)

print(
    customer_product_features[
        "share_of_customer_orders_containing_product"
    ].describe()
)

Customer-product reorder rate:
count    1.330795e+07
mean     2.655049e-01
std      3.397437e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      5.000000e-01
max      9.898990e-01
Name: customer_product_reorder_rate, dtype: float64

Customer order share:
count    1.330795e+07
mean     1.542015e-01
std      1.645884e-01
min      1.010101e-02
25%      4.545455e-02
50%      9.615385e-02
75%      2.000000e-01
max      1.000000e+00
Name: share_of_customer_orders_containing_product, dtype: float64


In [45]:
# ============================================================
# MISSING VALUES
# ============================================================

print("Customer features")
display(
    customer_features.isna().sum()
)

print("\nProduct features")
display(
    product_features.isna().sum()
)

print("\nCustomer × Product features")
display(
    customer_product_features.isna().sum()
)

print("\nCustomer × Department features")
display(
    customer_department_features.isna().sum()
)

Customer features


user_id                       0
total_orders                  0
avg_days_between_orders       0
median_days_between_orders    0
preferred_day_of_week         0
preferred_hour                0
unique_products_x             0
unique_aisles_x               0
unique_departments_x          0
avg_products_per_order_x      0
unique_products_y             0
unique_aisles_y               0
unique_departments_y          0
avg_products_per_order_y      0
customer_reorder_ratio        0
dtype: int64


Product features


product_id                 0
product_purchase_count     0
product_unique_users       0
product_reorder_rate       0
product_popularity_rank    0
product_name               0
aisle_id                   0
department_id              0
aisle                      0
department                 0
dtype: int64


Customer × Product features


user_id                                              0
product_id                                           0
customer_product_purchase_count                      0
customer_product_reorder_rate                        0
first_order_number                                   0
last_order_number                                    0
share_of_customer_orders_containing_product          0
customer_last_order_number                           0
orders_since_last_purchase                           0
avg_orders_between_purchases                   7982695
dtype: int64


Customer × Department features


user_id                               0
department_id                         0
customer_department_purchase_count    0
customer_total_product_purchases      0
customer_department_share             0
department                            0
dtype: int64

In [46]:
# ============================================================
# SAVE FEATURES
# ============================================================

customer_features.to_parquet(
    PROCESSED_DIR / "customer_features.parquet",
    index=False
)

product_features.to_parquet(
    PROCESSED_DIR / "product_features.parquet",
    index=False
)

customer_product_features.to_parquet(
    PROCESSED_DIR / "customer_product_features.parquet",
    index=False
)

customer_department_features.to_parquet(
    PROCESSED_DIR / "customer_department_features.parquet",
    index=False
)

print("✓ customer_features.parquet")
print("✓ product_features.parquet")
print("✓ customer_product_features.parquet")
print("✓ customer_department_features.parquet")

✓ customer_features.parquet
✓ product_features.parquet
✓ customer_product_features.parquet
✓ customer_department_features.parquet


In [47]:
# ============================================================
# VERIFY OUTPUT FILES
# ============================================================

for file in PROCESSED_DIR.glob("*.parquet"):

    print(
        f"{file.name:45} "
        f"{file.stat().st_size / 1024**2:.2f} MB"
    )

customer_department_features.parquet          12.41 MB
customer_features.parquet                     4.54 MB
customer_product_features.parquet             99.72 MB
product_features.parquet                      1.99 MB


In [48]:
# ============================================================
# CONSOLIDATED FEATURES
# ============================================================

features = (
    customer_product_features
    .merge(
        customer_features,
        on="user_id",
        how="left"
    )
    .merge(
        product_features[
            [
                "product_id",
                "product_purchase_count",
                "product_unique_users",
                "product_reorder_rate",
                "product_popularity_rank",
                "aisle_id",
                "department_id"
            ]
        ],
        on="product_id",
        how="left"
    )
)

print("Features shape:", features.shape)

display(features.head())

Features shape: (13307953, 30)


,user_id,product_id,customer_product_purchase_count,customer_product_reorder_rate,first_order_number,last_order_number,share_of_customer_orders_containing_product,customer_last_order_number,orders_since_last_purchase,avg_orders_between_purchases,total_orders,avg_days_between_orders,median_days_between_orders,preferred_day_of_week,preferred_hour,unique_products_x,unique_aisles_x,unique_departments_x,avg_products_per_order_x,unique_products_y,unique_aisles_y,unique_departments_y,avg_products_per_order_y,customer_reorder_ratio,product_purchase_count,product_unique_users,product_reorder_rate,product_popularity_rank,aisle_id,department_id
0,1,196,10,0.900000,1,10,1.0,10,0,1.0,10,19.555556,20.0,1,7,18,12,7,5.9,18,12,7,5.9,0.694915,35791,8000,0.776480,90.0,77,7
1,1,10258,9,0.888889,2,10,0.9,10,0,1.0,10,19.555556,20.0,1,7,18,12,7,5.9,18,12,7,5.9,0.694915,1946,557,0.713772,2871.0,117,19
2,1,10326,1,0.000000,5,5,0.1,10,5,NaN,10,19.555556,20.0,1,7,18,12,7,5.9,18,12,7,5.9,0.694915,5526,1923,0.652009,976.0,24,4
3,1,12427,10,0.900000,1,10,1.0,10,0,1.0,10,19.555556,20.0,1,7,18,12,7,5.9,18,12,7,5.9,0.694915,6476,1679,0.740735,830.0,23,19
4,1,13032,3,0.666667,2,10,0.3,10,0,4.0,10,19.555556,20.0,1,7,18,12,7,5.9,18,12,7,5.9,0.694915,3751,1286,0.657158,1512.0,121,14


In [49]:
# Ajouter l'affinité du client au département

features = features.merge(
    customer_department_features[
        [
            "user_id",
            "department_id",
            "customer_department_purchase_count",
            "customer_total_product_purchases",
            "customer_department_share"
        ]
    ],
    on=["user_id", "department_id"],
    how="left"
)

print("Final features shape:", features.shape)

Final features shape: (13307953, 33)


In [50]:
# ============================================================
# SAVE CONSOLIDATED FEATURES
# ============================================================

features.to_parquet(
    PROCESSED_DIR / "features.parquet",
    index=False
)

print(
    "✓ Saved:",
    PROCESSED_DIR / "features.parquet"
)

✓ Saved: C:\Users\djuff\OneDrive\Drive_perso_ceed\OneDrive\Documents\CDI\instacart_propensity\data\processed\features.parquet


In [51]:
# ============================================================
# FINAL CHECK
# ============================================================

print("=" * 60)
print("FEATURE ENGINEERING COMPLETED")
print("=" * 60)

print("\nShape:")
print(features.shape)

print("\nNumber of customers:")
print(features["user_id"].nunique())

print("\nNumber of products:")
print(features["product_id"].nunique())

print("\nNumber of customer-product pairs:")
print(
    features[
        ["user_id", "product_id"]
    ].drop_duplicates().shape[0]
)

print("\nOutput directory:")
print(PROCESSED_DIR)

print("\nFiles:")
for file in PROCESSED_DIR.glob("*.parquet"):
    print(" -", file.name)

FEATURE ENGINEERING COMPLETED

Shape:
(13307953, 33)

Number of customers:
206209

Number of products:
49677

Number of customer-product pairs:
13307953

Output directory:
C:\Users\djuff\OneDrive\Drive_perso_ceed\OneDrive\Documents\CDI\instacart_propensity\data\processed

Files:
 - customer_department_features.parquet
 - customer_features.parquet
 - customer_product_features.parquet
 - features.parquet
 - product_features.parquet
